# Feature Reduction using Genetic Algorithm

Genetic algorithms, inspired by the process of natural selection, are optimization techniques that **mimic biological evolution** to solve complex problems. In a genetic algorithm (GA), a population of potential solutions is iteratively improved by applying mechanisms such as **selection**, **crossover**, and **mutation**. Initially, a diverse set of solutions, represented as **chromosomes**, is generated randomly. Through successive generations, better solutions emerge as individuals with favorable traits are selected to produce **offspring**. **Crossover** involves **combining genetic material from two parent solutions** to create new ones, while **mutation** introduces **random changes to maintain diversity**.

For feature reduction, genetic algorithms can identify the most informative subset of features from a larger set. In this approach, each chromosome in the population represents a candidate feature subset, and fitness is evaluated using a performance metric—such as classification accuracy or F1-score. Through iterative evolution, the algorithm favors individuals that enhance predictive power while penalizing redundancy or irrelevant features. This results in a reduced feature set that improves both model efficiency and interpretability.

The illustration below demonstrates a classification task before and after applying a genetic algorithm (via the `pymoo` package) to reduce features in a randomly generated sample dataset. The objective function for population selection is the F1-score, which we aim to optimize. If you haven't yet installed the `pymoo` package, you can do so by running `pip install pymoo`. For more information on the `pymoo` package and GA, visit: https://pymoo.org/ and https://pymoo.org/algorithms/soo/ga.html#nb-ga.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

# Load the sample dataset
data = pd.read_csv("sample_data.csv")

X = data.drop(columns = ['target'])
y = data['target']

# Split the dataset into training and testing sets - Training 70% and Testing 30%
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=30)

# Train and predict using a Gaussian Naive Bayes classifier
model_gb=GaussianNB()
model_gb.fit(X_train,y_train)
gb_pred = model_gb.predict(X_test)

# Calculate Confusion Matrix
conf_matrix = confusion_matrix(y_test, gb_pred)

# Calculate Accuracy
accuracy = accuracy_score(y_test, gb_pred)

# Calculate Precision
precision = precision_score(y_test, gb_pred)

# Calculate Recall
recall = recall_score(y_test, gb_pred)

# Calculate F1 Score
f1 = f1_score(y_test, gb_pred)

# Print the results
print("\nGaussian Naive Bayes Performance")
print("==================================\n")
print(f"Confusion Matrix:\n{conf_matrix}")
print(f"\n Accuracy: {accuracy}")
print(f"\n Precision: {precision}")
print(f"\n Recall: {recall}")
print(f"\n F1 Score: {f1}")

# Train and predict using the Logistic Regression classifier
model_lr = LogisticRegression(solver='lbfgs', max_iter=500)
model_lr.fit(X_train, y_train)
lr_pred = model_lr.predict(X_test)

# Calculate Confusion Matrix
conf_matrix = confusion_matrix(y_test, lr_pred)

# Calculate Accuracy
accuracy = accuracy_score(y_test, lr_pred)

# Calculate Precision
precision = precision_score(y_test, lr_pred)

# Calculate Recall
recall = recall_score(y_test, lr_pred)

# Calculate F1 Score
f1 = f1_score(y_test, lr_pred)

# Print the results
print("\n Logistic Regression Performance")
print("==================================\n")
print(f"Confusion Matrix:\n{conf_matrix}")
print(f"\n Accuracy: {accuracy}")
print(f"\n Precision: {precision}")
print(f"\n Recall: {recall}")
print(f"\n F1 Score: {f1}")

**Feature Reduction using Genetic Algorithm**

In [ ]:
import numpy as np
import pandas as pd

# ==========================================================
# Import pymoo components for optimization
# ==========================================================
from pymoo.core.problem import Problem
from pymoo.algorithms.soo.nonconvex.ga import GA
from pymoo.core.problem import Problem
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.operators.repair.rounding import RoundingRepair
from pymoo.operators.sampling.rnd import IntegerRandomSampling
from pymoo.optimize import minimize

# ==========================================================
# Import machine learning models and evaluation metrics
# ==========================================================
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import f1_score

# Load the sample dataset
data = pd.read_csv("sample_data.csv")

X = data.drop(columns = ['target'])
y = data['target']

 # Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=30)

# ==========================================================
# Single-objective Feature Selection Problem
# ==========================================================
class FeatureSelectionProblem(Problem):

    def __init__(self):
        n = len(X.columns)
        super().__init__(n_var=n, n_obj=1, vtype=int, xl=0.0, xu=1.0)

    def __evaluate_one(self, x) -> float:
        
        XX_train = X_train.iloc[:, x==1]
        XX_test = X_test.iloc[:, x==1]
        # Create and train the Gaussian Naive Bayes classifier
        model_gb=GaussianNB()
        model_gb.fit(XX_train,y_train)
        gb_pred = model_gb.predict(XX_test)
        # Calculate F1 Score 
        return f1_score(y_test, gb_pred)

    def _evaluate(self, x, out, *args, **kwargs):
        objectives = np.zeros(len(x))
        for i, _x in enumerate(x):
            objectives[i] = -self.__evaluate_one(_x)
        out['F'] = objectives
              
problem = FeatureSelectionProblem()

# ==========================================================
# GA Algorithm
# ==========================================================
algorithm = GA(pop_size=100,
            sampling=IntegerRandomSampling(),
            crossover=SBX(prob=1.0, eta=3.0, vtype=float, repair=RoundingRepair()),
            mutation=PM(prob=1.0, eta=3.0, vtype=float, repair=RoundingRepair()),
            eliminate_duplicates=True,
            )

res = minimize(problem,
               algorithm,
               termination=('n_gen', 10),
               seed=5,
               save_history=True,
               verbose=True
               )

In [ ]:
# Display the optimal solution
# res.F[0] is the best (negative) F1 value from minimization
best_f1 = -res.F[0]          # convert back to positive F
print(f"Best F1 value: = {best_f1:.4f}")

# res.X is the best chromosome (1D array)
selected_features = X.columns[res.X == 1]
print("Number of features:", len(selected_features))
print("Selected features:", list(selected_features))

# Feature Reduction using Multi-objective Genetic Algorithm

Multi-objective optimization is a technique used to optimize two or more conflicting objectives simultaneously. Instead of finding a single optimal solution, the algorithm searches for a set of optimal trade-off solutions called the **Pareto Front** (non-dominated solutions).

In this feature selection problem:

- **Objective 1:** Maximize the F1 Score  
- **Objective 2:** Minimize the number of selected features  

Since improving classification performance may require more features, these objectives conflict with each other. Therefore, the optimization algorithm attempts to find feature subsets that provide the best balance between model performance and feature reduction.

**Pareto Front and Non-Dominated Solutions**

For example, in feature selection:

- **Objective 1:** Maximize F1 Score  
- **Objective 2:** Minimize the number of selected features  

A solution is considered non-dominated when:

- No other solution has a **higher F1 score** and  
- At the same time uses **fewer features**.

The collection of all non-dominated solutions forms the **Pareto Front**.

Each point on the Pareto Front represents an optimal trade-off between the objectives. Improving one objective usually causes degradation in the other objective. Therefore, instead of one single best solution, the Pareto Front provides multiple balanced solutions for decision-making.

**Example**

| Solution | F1 Score | No. of Features | Status |
|----------|-----------|----------------|--------|
| A | 0.95 | 25 | Non-dominated |
| B | 0.93 | 15 | Non-dominated |
| C | 0.90 | 10 | Non-dominated |
| D | 0.90 | 20 | Dominated |

Solution **D** is dominated because Solution **C** achieves the same F1 score using fewer features.

Thus, the Pareto Front contains only the best trade-off solutions, known as non-dominated solutions.

The optimization is performed using the **NSGA-II (Non-dominated Sorting Genetic Algorithm II)** algorithm, which evolves multiple candidate solutions over generations and identifies non-dominated solutions forming the Pareto Front.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# ==========================================================
# Import pymoo components for optimization
# ==========================================================
from pymoo.core.problem import Problem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.operators.repair.rounding import RoundingRepair
from pymoo.operators.sampling.rnd import IntegerRandomSampling
from pymoo.optimize import minimize
from pymoo.core.callback import Callback
from pymoo.util.nds.non_dominated_sorting import NonDominatedSorting

# ==========================================================
# Import machine learning models and evaluation metrics
# ==========================================================
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import f1_score

data = pd.read_csv("sample_data.csv")

X = data.drop(columns=['target'])
y = data['target']

 # Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=30)

# ==========================================================
# Bi-objective Feature Selection Problem
# ==========================================================
class FeatureSelectionProblem(Problem):

    def __init__(self):

        n = len(X.columns)

        super().__init__(
            n_var=n,
            n_obj=2,
            n_constr=1,   # number of constraints
            vtype=int,
            xl=0,
            xu=1
        )

    def __evaluate_one(self, x):

        if np.sum(x) == 0:
            return 0.0

        XX_train = X_train.iloc[:, x == 1]
        XX_test = X_test.iloc[:, x == 1]

        model_gb = GaussianNB()
        model_gb.fit(XX_train, y_train)

        gb_pred = model_gb.predict(XX_test)

        return f1_score(y_test, gb_pred)

    def _evaluate(self, x, out, *args, **kwargs):

        F = []
        G = []   # constraints

        for _x in x:

            f1 = self.__evaluate_one(_x)

            # objectives
            obj1 = -f1
            obj2 = np.sum(_x)

            F.append([obj1, obj2])

            # constraint: number of features must be > 30
            g1 = 30 - np.sum(_x)   # <= 0 means valid
            G.append([g1])

        out["F"] = np.array(F)
        out["G"] = np.array(G)

# ==========================================================
# Callback:
# Display ONLY non-dominated objective values
# ==========================================================
class MyCallback(Callback):

    def notify(self, algorithm):

        # Clear screen every generation
        os.system('cls' if os.name == 'nt' else 'clear')

        gen = algorithm.n_gen

        F = algorithm.pop.get("F")

        # Get non-dominated front indices
        nds = NonDominatedSorting()
        front = nds.do(F, only_non_dominated_front=True)

        print(f"\nGeneration: {gen}")
        print("=" * 50)
        print("NON-DOMINATED OBJECTIVE VALUES")
        print("=" * 50)

        print("\n   F1 Score\t\tNo. Features")
        print("-" * 50)

        for idx in front:

            f1 = -F[idx][0]           # Convert back to positive F1
            n_features = int(F[idx][1])

            print(f"   {f1:.4f}\t\t{n_features}")

# ==========================================================
# Problem
# ==========================================================
problem = FeatureSelectionProblem()

# ==========================================================
# NSGA-II Algorithm
# ==========================================================
algorithm = NSGA2(
    pop_size=100,
    sampling=IntegerRandomSampling(),

    crossover=SBX(
        prob=1.0,
        eta=3.0,
        vtype=float,
        repair=RoundingRepair()
    ),

    mutation=PM(
        prob=1.0,
        eta=3.0,
        vtype=float,
        repair=RoundingRepair()
    ),

    eliminate_duplicates=True,
)

# ==========================================================
# Optimization
# ==========================================================
res = minimize(
    problem,
    algorithm,
    termination=('n_gen', 10),
    seed=5,
    save_history=True,
    verbose=False,
    callback=MyCallback()
)

# ==========================================================
# FINAL SOLUTIONS ONLY
# ==========================================================
print("\n\nFINAL PARETO OPTIMAL SOLUTIONS")
print("=" * 60)

for i in range(len(res.F)):

    f1 = -res.F[i][0]
    n_features = int(res.F[i][1])

    selected_features = X.columns[res.X[i] == 1]

    print(f"\nSolution {i+1}")
    print("-" * 40)
    print(f"F1 Score           : {f1:.4f}")
    print(f"Number of Features : {n_features}")
    print("Selected Features  :", list(selected_features))

# ==========================================================
# Plot Pareto Front
# ==========================================================
pareto_f1 = [-f[0] for f in res.F]
pareto_features = [f[1] for f in res.F]

plt.figure(figsize=(8, 6))

plt.scatter(
    pareto_features,
    pareto_f1,
    color='red',
    s=80
)

plt.xlabel("Number of Selected Features")
plt.ylabel("F1 Score")
plt.title("Pareto Front")
plt.grid(True)

plt.show()